# Documentation Helper - Core RAG Pipeline (Colab)

This notebook loads the local Chroma vector store produced by the ingestion notebook and runs a tool-enabled LangChain agent for question answering.

It retrieves relevant documentation context first and then generates grounded answers with source-aware context artifacts.


In [1]:
%pip -q install   certifi   python-dotenv   chromadb   langchain-core   langchain-classic   langchain-chroma   langchain-openai   langchain-tavily   nest_asyncio


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.0/23.0 MB 52.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 47.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 67.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 78.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 66.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 587.6/587

In [15]:
import os
from typing import Any, Dict
from dotenv import load_dotenv

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.messages import ToolMessage
from langchain.tools import tool
from langchain_pinecone import PineconeVectorStore
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

In [17]:
# Initialize embeddings (same as ingestion.py)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# Initialize local Chroma vector store.
# Make sure ingestion ran first and populated this directory.
vectorstore = Chroma(persist_directory="chroma_db", embedding_function=embeddings)

# Initialize chat model
model = init_chat_model("gpt-5.2", model_provider="openai")


In [18]:
@tool(response_format="content_and_artifact")
def retrieve_context(query: str):
    """Retrieve relevant documentation to help answer user queries about LangChain."""
    retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
    retrieved_docs = retriever.invoke(query)

    serialized = "\n\n".join(
        (
            f"Source: {doc.metadata.get('source', 'Unknown')}\n\n"
            f"Content: {doc.page_content}"
        )
        for doc in retrieved_docs
    )
    return serialized, retrieved_docs

In [19]:

def run_llm(query: str) -> Dict[str, Any]:
    """
    Run the RAG pipeline to answer a query using retrieved documentation.

    Args:
        query: The user's question.

    Returns:
        Dictionary containing:
            - answer: The generated answer.
            - context: List of retrieved documents.
    """
    system_prompt = (
        "You are a helpful AI assistant that answers questions about LangChain documentation. "
        "You have access to a tool that retrieves relevant documentation. "
        "Use the tool to find relevant information before answering questions. "
        "Always cite the sources you use in your answers. "
        "If you cannot find the answer in the retrieved documentation, say so."
    )

    agent = create_agent(model, tools=[retrieve_context], system_prompt=system_prompt)
    response = agent.invoke({"messages": [{"role": "user", "content": query}]})
    answer = response["messages"][-1].content

    context_docs = []

    for message in response["messages"]:
        if isinstance(message, ToolMessage) and hasattr(message, "artifact"):
            if isinstance(message.artifact, list):
                context_docs.extend(message.artifact)

    return {"answer": answer, "context": context_docs}

In [20]:
run_llm(query="what are deep agents?")

{'answer': 'Deep Agents are LangChain’s “batteries-included” agent implementations (a higher-level SDK) designed for building robust agents that can tackle complex, long-running, multi-step work with better context and execution management than a minimal agent setup. They’re recommended as the starting point if you’re building an agent and want these built-ins. [LangChain overview](https://python.langchain.com/) • [Deep Agents overview](https://docs.langchain.com/oss/python/deepagents/overview)\n\nKey capabilities highlighted in the docs:\n\n- **Planning & task decomposition** (e.g., a built-in `write_todos` tool to break work into steps and track progress). [Deep Agents overview](https://docs.langchain.com/oss/python/deepagents/overview)\n- **Context management at scale**, including a **virtual filesystem** (`ls`, `read_file`, `write_file`, `edit_file`) to offload/organize large context and **auto-summarization** to compress long conversations. [Deep Agents overview](https://docs.lang